In [1]:
import mina
import scanpy as sc
import scanpy.external as sce
import numpy as np
import decoupler as dc
import math
import mofaflex as mf
import mudata as md
import pandas as pd
import matplotlib.pyplot as plt
import squidpy as sq

/nfs/research/saezrodriguez/rico/MINA_ms/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/nfs/research/saezrodriguez/rico/MINA_ms/.venv/lib/python3.12/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/nfs/research/saezrodriguez/rico/MINA_ms/.venv/lib/python3.12/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/nfs/research/saezrodriguez/rico/MINA_ms/.venv/lib/python3.12/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Versio

Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



In [2]:
adata = sc.read_h5ad("/nfs/research/saezrodriguez/data/ERC_CartoHostBug/MERFISH/full_adata.h5ad")

/tmp/ipykernel_3756193/3362090997.py:1: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.


In [3]:
# add obs names
adata.obs_names = adata.obs["Cell_ID"].to_list()

In [4]:
# Generating sample meta-data
metadata = mina.up.extract_metadata_from_obs(obs=adata.obs, groupby="Slice_ID", sort=False)
metadata["Sample_type"] = pd.Categorical(metadata["Sample_type"], categories=["Healthy", "DSS3", "DSS9", "DSS21", "DSS35"], ordered=True)

In [5]:
#Define order of time points
pdata = dc.pp.pseudobulk(adata, sample_col="Slice_ID", groups_col="Tier1")
# Manipulations on pdata
pdata.obs = pdata.obs.set_index("Slice_ID", drop=False)
pdata.obs.index.name = None

# Make list of adatas per pseudobulk cell type
anndata_dict = mina.up.split_anndata_by_celltype(pdata=pdata, grouping="Tier1")

# Adding gene expression total counts - TODO add to upstream as a function
for _cell_type, adata_raw in anndata_dict.items():
    # Sum across observations (rows) for each gene (column)
    if hasattr(adata_raw.X, "toarray"):
        # Sparse matrix case
        total_counts = adata_raw.X.sum(axis=0).A1  # returns 1D array
    else:
        total_counts = adata_raw.X.sum(axis=0)  # numpy array

    # Add to .var
    adata_raw.var["total_counts"] = total_counts

mina.up.filt.filter_anndata_by_ncells(anndata_dict, min_cells=10)
# From all the possible samples, let's ask for at least 40%
n_samples = len(metadata.index)
min_samples =  math.trunc((n_samples * 0.9))
mina.up.filter_views_by_samples(anndata_dict, min_rows=min_samples)
mina.up.filter_genes_byexpr(anndata_dict, min_count=5, min_prop=0.4)
mina.up.filter_views_by_genes(anndata_dict, min_genes_per_view=50)
mina.up.filter_samples_by_coverage(anndata_dict, threshold=0, min_prop=0.95)
mina.up.filter_views_by_samples(anndata_dict, min_rows=min_samples)
mina.up.save_raw_counts(anndata_dict, layer_name="raw_counts")
mina.up.norm_log(anndata_dict, target_sum=1e5, exclude_highly_expressed=False, max_value=None, center=True)

Raw counts saved in the 'raw_counts' layer for each AnnData object.
Normalization, log-transformation, and scaling complete for all AnnData objects with target_sum = 100000.0.


/tmp/ipykernel_3756193/1031412037.py:4: UserWarning: 
AnnData expects .obs.index to contain strings, but got values like:
    ['062221_D9_m3_2_slice_1', '062221_D9_m3_2_slice_2', '062221_D9_m3_2_slice_3', '062921_D0_m3a_1_slice_1', '062921_D0_m3a_1_slice_2']

    Inferred to be: categorical



In [6]:
anndata_dict

{'Endothelial': AnnData object with n_obs × n_vars = 58 × 984
     obs: 'Slice_ID', 'Tier1', 'Mouse_ID', 'Technical_repeat_number', 'Sample_type', 'dataset', 'psbulk_cells', 'psbulk_counts'
     var: 'total_counts', 'mean', 'std'
     uns: 'log1p'
     layers: 'psbulk_props', None (.X), 'raw_counts',
 'EntericNervous': AnnData object with n_obs × n_vars = 57 × 929
     obs: 'Slice_ID', 'Tier1', 'Mouse_ID', 'Technical_repeat_number', 'Sample_type', 'dataset', 'psbulk_cells', 'psbulk_counts'
     var: 'total_counts', 'mean', 'std'
     uns: 'log1p'
     layers: 'psbulk_props', None (.X), 'raw_counts',
 'Epithelial': AnnData object with n_obs × n_vars = 58 × 987
     obs: 'Slice_ID', 'Tier1', 'Mouse_ID', 'Technical_repeat_number', 'Sample_type', 'dataset', 'psbulk_cells', 'psbulk_counts'
     var: 'total_counts', 'mean', 'std'
     uns: 'log1p'
     layers: 'psbulk_props', None (.X), 'raw_counts',
 'Fibroblast': AnnData object with n_obs × n_vars = 58 × 987
     obs: 'Slice_ID', 'Tier1', 

In [7]:
# Calculate correlation between observations for each anndata in the dictionary

cor_samples = {}

for cell_type, ad in anndata_dict.items():
    # Calculate correlation matrix
    corr_matrix = np.corrcoef(ad.X)  # Transpose to get genes as rows
    # Make a dataframe adding the obs names as index and columns and make it long format
    corr_df = pd.DataFrame(corr_matrix, index=ad.obs_names, columns=ad.obs_names)
    corr_df_long = corr_df.reset_index().melt(id_vars='index', var_name='sample2', value_name='correlation')
    # Filter out self-correlations (where gene1 == gene2)
    corr_df_long = corr_df_long[corr_df_long['index'] != corr_df_long['sample2']]
    # Add sample information from metadata
    corr_df_long = corr_df_long.merge(metadata[["Slice_ID","Sample_type"]], left_on='index', right_on='Slice_ID', how='left')
    # Rename the sample_type column to sample 1
    corr_df_long = corr_df_long.rename(columns={"Sample_type": "Sample_type_sample1"})

    # Add sample information from metadata
    corr_df_long = corr_df_long.merge(metadata[["Slice_ID","Sample_type"]], left_on='sample2', right_on='Slice_ID', how='left')
    # Rename the sample_type column to sample 1
    corr_df_long = corr_df_long.rename(columns={"Sample_type": "Sample_type_sample2"})
    corr_df_long = corr_df_long[corr_df_long['Sample_type_sample2'] == corr_df_long['Sample_type_sample1']]
    # Take the mean group by Sample_type, summarize
    corr_df_long = corr_df_long.groupby(['Sample_type_sample1']).agg({'correlation': 'mean'}).reset_index()
    cor_samples[cell_type] = corr_df_long

# Bind rows of all dataframes in cor_samples into a single dataframe
cor_samples_df = pd.concat(cor_samples, names=['cell_type', 'index']).reset_index()
cor_samples_df = cor_samples_df.rename(columns={"index": "Sample_type", "correlation": "mean_correlation"})
cor_samples_df = cor_samples_df.groupby(['Sample_type_sample1']).agg({'mean_correlation': 'mean'}).reset_index()

print(cor_samples_df)



  Sample_type_sample1  mean_correlation
0             Healthy          0.271102
1                DSS3          0.410165
2                DSS9          0.159007
3               DSS21          0.035456
4               DSS35          0.504121


/tmp/ipykernel_3756193/4094410525.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
/tmp/ipykernel_3756193/4094410525.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
/tmp/ipykernel_3756193/4094410525.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
/tmp/ipykernel_3756193/4094410525.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass obser